# Full-Mamba HADA — Zhou2016 LOSO
Enable **GPU** and **Internet** in Kaggle, then run the cell below. Each MI trial uses the native five-second cue interval `[0, 5.0 s)`.

In [ ]:
# One-cell Kaggle: Zhou2016 LOSO targets S01-S04. Enable GPU + Internet.
import codecs, os, select, shutil, subprocess, sys, time
from pathlib import Path

BRANCH = 'feature/hada-full-mamba-zhou2016'
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
REPO_PATH = Path('/kaggle/working/tcformer-full-mamba-zhou2016')
SUBJECT_IDS = list(range(1, 5))
MAX_EPOCHS = 100
BATCH_SIZE = 32
VALIDATION_FRACTION = 0.20
USE_RA = True
USE_IM_TTA = True
IM_TTA_STEPS = 5
if USE_IM_TTA and IM_TTA_STEPS < 1:
    raise ValueError('IM_TTA_STEPS must be positive when USE_IM_TTA=True')
RUN_TAG = f'ra-{int(USE_RA)}_imtta-{IM_TTA_STEPS if USE_IM_TTA else 0}'
MNE_DATA = Path('/kaggle/working/mne_data')
RESULT_ARCHIVE = Path(f'/kaggle/working/full_mamba_zhou2016_loso_5s_ep{MAX_EPOCHS}_{RUN_TAG}_results')
TRAIN_LOG = Path(f'/kaggle/working/full_mamba_zhou2016_loso_5s_ep{MAX_EPOCHS}_{RUN_TAG}.log')

def run(command, cwd=None, env=None, stream=False, log_path=None):
    command = list(map(str, command))
    print('+', ' '.join(command), flush=True)
    if not stream:
        subprocess.run(command, cwd=str(cwd) if cwd else None, env=env, check=True)
        return
    process = subprocess.Popen(command, cwd=str(cwd) if cwd else None, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    started_at = time.monotonic()
    with open(log_path, 'w', encoding='utf-8') as log_file:
        while True:
            ready, _, _ = select.select([process.stdout], [], [], 30)
            if ready:
                chunk = os.read(process.stdout.fileno(), 4096)
                if not chunk:
                    break
                output = decoder.decode(chunk)
                print(output, end='', flush=True)
                log_file.write(output)
                log_file.flush()
            elif process.poll() is None:
                heartbeat = f'[Kaggle heartbeat] process is running | elapsed={(time.monotonic() - started_at) / 60:.1f}m\n'
                print(heartbeat, end='', flush=True)
                log_file.write(heartbeat)
                log_file.flush()
            else:
                break
        remaining = decoder.decode(b'', final=True)
        if remaining:
            print(remaining, end='', flush=True)
            log_file.write(remaining)
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'Command failed with exit code {return_code}. Full log: {log_path}')

run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
if (REPO_PATH / '.git').is_dir():
    run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_PATH)
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_PATH)
    # A previous run modifies the YAML config locally. Force the disposable
    # Kaggle clone back to the fetched branch so rerunning this cell is safe.
    run(['git', 'checkout', '--force', BRANCH], cwd=REPO_PATH)
    run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=REPO_PATH)
else:
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_PATH])
run(['git', 'log', '-1', '--oneline'], cwd=REPO_PATH)

UV = shutil.which('uv') or 'uv'
run([UV, 'venv', '--clear', '--python', '3.10', '.venv'], cwd=REPO_PATH)
PYTHON = REPO_PATH / '.venv/bin/python'
run([UV, 'pip', 'install', '--python', PYTHON, 'torch==2.7.1', 'torchvision==0.22.1', '--index-url', 'https://download.pytorch.org/whl/cu126'])
run([UV, 'pip', 'install', '--python', PYTHON, '-r', 'requirements.txt'], cwd=REPO_PATH)

override = f"import yaml; from pathlib import Path; p=Path('configs/hada_tcformer.yaml'); c=yaml.safe_load(p.read_text()); c['subject_ids']={SUBJECT_IDS!r}; c['max_epochs_loso_zhou2016']={MAX_EPOCHS}; pre=c['preprocessing']['zhou2016']; pre['start']=0.0; pre['trial_duration']=5.0; pre['batch_size']={BATCH_SIZE}; pre['validation_fraction']={VALIDATION_FRACTION}; pre['riemannian_alignment']={USE_RA!r}; pre['interaug']=False; pre.setdefault('model_overrides', {{}}).update({{'lr': 0.001, 'weight_decay': 0.0001}}); c['model_kwargs']['im_tta_steps']={IM_TTA_STEPS if USE_IM_TTA else 0}; p.write_text(yaml.safe_dump(c, sort_keys=False))"
run([PYTHON, '-c', override], cwd=REPO_PATH)
run(['nvidia-smi'])
run([PYTHON, '-c', "import torch; print('PyTorch:', torch.__version__); print('CUDA:', torch.cuda.is_available()); assert torch.cuda.is_available(), 'Kaggle GPU is not enabled'; print('GPU:', torch.cuda.get_device_name(0))"])

MNE_DATA.mkdir(parents=True, exist_ok=True)
environment = os.environ.copy()
environment.update({'PYTHONUNBUFFERED': '1', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True', 'MPLBACKEND': 'Agg', 'MNE_DATA': str(MNE_DATA), 'MNE_DATASETS_ZHOU2016_PATH': str(MNE_DATA)})
print('===== FULL-MAMBA HADA | ZHOU2016 LOSO S01-S04 | 3 CLASSES | 5.0 s | 1250 SAMPLES =====', flush=True)
print(f'Options: epochs={MAX_EPOCHS} | batch={BATCH_SIZE} | source val={VALIDATION_FRACTION:.0%} | lr=0.001 | weight_decay=0.0001 | RA={USE_RA} | IM-TTA steps={IM_TTA_STEPS if USE_IM_TTA else 0} | InterAug=False', flush=True)
print('Protocol: all 3 source sessions=source-only stratified 80/20 train/validation; all 3 target sessions=unlabeled HADA + IM-TTA, then test', flush=True)
print('Live log:', TRAIN_LOG, flush=True)
run([PYTHON, '-u', 'train_pipeline.py', '--model', 'hada_tcformer', '--dataset', 'zhou2016', '--loso', '--gpu_id', '0', '--no_interaug'], cwd=REPO_PATH, env=environment, stream=True, log_path=TRAIN_LOG)
archive = shutil.make_archive(str(RESULT_ARCHIVE), 'zip', root_dir=REPO_PATH, base_dir='results')
print('Zhou2016 LOSO complete. Kaggle output:', archive, flush=True)
